In [ ]:
# Установка gdown для скачивания с Google Drive
!pip install -q gdown


In [ ]:
# Установка зависимостей (минимально, без переустановки torch)
!pip install -q -U pip
!pip install -q -U transformers accelerate sentencepiece tqdm tabulate pynvml psutil
# Принудительное обновление bitsandbytes для поддержки квантования
!pip install -q -U bitsandbytes
# Проверка версии
!pip show bitsandbytes
# Опционально (может не собраться на Windows)
# %pip install -q xformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 85.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ucxx-cu12 0.44.0 requires pynvml<13.0.0a0,>=12.0.0, but you have pynvml 13.0.1 which is incompatible.
dask-cuda 25.6.0 requires pynvml<13.0.0a0,>=12.0.0, but you have pynvml 13.0.1 which is incompatible.
ucx-py-cu12 0.44.0 requires pynvml<13.0.0a0,>=12.0.0, but you have pynvml 13.0.1 which is incompatible.
dask-cudf-cu12 25.6.0 requires pynvml<13.0.0a0,>=12.0.0, but you have pynvml 13.0.1 which is incompatible.
Name: bitsandbytes
Version: 0.48.1
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License-Expression: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy, packaging, to

In [ ]:
# === Загрузка RAG-артефактов с Google Drive ===
import gdown
import os
from pathlib import Path

# Создаем папку для артефактов
RAG_DIR = Path('rag')
RAG_DIR.mkdir(exist_ok=True)

# URL'ы файлов на Google Drive
urls = {
    "meta.json": "https://drive.google.com/uc?id=1L_x6ifbfRH2YboGS8OKp95uXM7Wkz8zG",
    "index.faiss": "https://drive.google.com/uc?id=1-grB4MbvNYZIGZm7qMVwLnu2xWM9PjTt",
    "chunks.parquet": "https://drive.google.com/uc?id=1-ZEmZ9kRT73FQx8c23Kz_BHmlhXuHvDA"
}

# Скачиваем каждый файл
for filename, url in urls.items():
    output_path = RAG_DIR / filename
    print(f"Скачивание {filename}...")
    gdown.download(url, str(output_path), quiet=False)

# Проверяем содержимое папки
print("\nСодержимое папки rag:")
!ls -lh {RAG_DIR}


Скачивание meta.json...


Downloading...
From: https://drive.google.com/uc?id=1L_x6ifbfRH2YboGS8OKp95uXM7Wkz8zG
To: /content/rag/meta.json
100%|██████████| 119/119 [00:00<00:00, 486kB/s]


Скачивание index.faiss...


Downloading...
From (original): https://drive.google.com/uc?id=1-grB4MbvNYZIGZm7qMVwLnu2xWM9PjTt
From (redirected): https://drive.google.com/uc?id=1-grB4MbvNYZIGZm7qMVwLnu2xWM9PjTt&confirm=t&uuid=4c064690-728a-46d4-ace2-6b0dc81c6195
To: /content/rag/index.faiss
100%|██████████| 7.70G/7.70G [03:22<00:00, 38.0MB/s]


Скачивание chunks.parquet...


Downloading...
From (original): https://drive.google.com/uc?id=1-ZEmZ9kRT73FQx8c23Kz_BHmlhXuHvDA
From (redirected): https://drive.google.com/uc?id=1-ZEmZ9kRT73FQx8c23Kz_BHmlhXuHvDA&confirm=t&uuid=dbd575a8-9fea-4070-9d9b-b30edc5a09d4
To: /content/rag/chunks.parquet
100%|██████████| 4.11G/4.11G [01:33<00:00, 44.2MB/s]



Содержимое папки rag:
total 12G
-rw-r--r-- 1 root root 3.9G Oct 28 10:59 chunks.parquet
-rw-r--r-- 1 root root 7.2G Oct 28 11:18 index.faiss
-rw-r--r-- 1 root root  119 Oct 28 10:49 meta.json


In [ ]:
# Проверка GPU и окружения
import torch, platform, psutil
import pynvml

pynvml.nvmlInit()
h = pynvml.nvmlDeviceGetHandleByIndex(0)
name = pynvml.nvmlDeviceGetName(h)
if isinstance(name, bytes):
    name = name.decode()
mem = pynvml.nvmlDeviceGetMemoryInfo(h)
driver = pynvml.nvmlSystemGetDriverVersion()
if isinstance(driver, bytes):
    driver = driver.decode()
gpu_info = {
    'name': name,
    'total_GB': round(mem.total/1024**3, 2),
    'driver': driver
}


print({'python': platform.python_version(),
       'torch': torch.__version__,
       'cuda_available': torch.cuda.is_available(),
       'device_count': torch.cuda.device_count(),
       'gpu_info': gpu_info})


{'python': '3.12.12', 'torch': '2.8.0+cu126', 'cuda_available': True, 'device_count': 1, 'gpu_info': {'name': 'NVIDIA L4', 'total_GB': 22.49, 'driver': '550.54.15'}}


In [ ]:
# Загрузка модели Saiga YandexGPT 8B (int8, с квантованием)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig

MODEL_NAME = "IlyaGusev/saiga_yandexgpt_8b"

# Паддинг-токен (часто отсутствует явно)
DEFAULT_PAD_TOKEN = "<|endoftext|>"


def _safe_gen_config(model_name: str):
    try:
        return GenerationConfig.from_pretrained(model_name)
    except Exception:
        # Минимальный фолбэк-конфиг
        return GenerationConfig(
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            top_k=0,
            top_p=1.0,
            repetition_penalty=1.0,
            eos_token_id=None,
            pad_token_id=None,
        )


def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': DEFAULT_PAD_TOKEN})
    gen_cfg = _safe_gen_config(MODEL_NAME)

    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        offload_folder="offload",
        trust_remote_code=True,
    )
    model.resize_token_embeddings(len(tokenizer))
    model.eval()

    precision = 'int8'
    return model, tokenizer, gen_cfg, precision

model, tokenizer, generation_config, load_mode = load_model()
print({'load_mode': load_mode, 'device': str(model.device), 'dtype': model.dtype})


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

{'load_mode': 'int8', 'device': 'cuda:0', 'dtype': torch.float16}


In [ ]:
SYSTEM_PROMPT = (
    "Ты — лаконичный и полезный ассистент на русском языке.\n\n"
    "ПРАВИЛА:\n"
    "1. Отвечай кратко и по делу.\n"
    "2. Если ответа не знаешь — отвечай: 'Не знаю'.\n"
    "3. Можно использовать общеизвестные факты без излишней осторожности.\n"
    "4. Переводы — только перевод без кавычек и комментариев.\n"
)

In [ ]:
import time
from statistics import mean
from tqdm.auto import tqdm

In [ ]:
import re

def build_prompt(user_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text}
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        # Fallback (плоский промпт)
        return f"[SYSTEM]\n{SYSTEM_PROMPT}\n\n[USER]\n{user_text}\n\n[ASSISTANT] "


def _normalize_text(query: str, text: str) -> str:
    t = (text or "").strip()
    if not t:
        return "Не знаю"

    # Аккуратно обрежем до первого завершённого предложения, если ответ длинный
    first_end = len(t)
    for sep in ".!?":
        pos = t.find(sep)
        if pos != -1:
            first_end = min(first_end, pos + 1)
    if first_end != len(t) and first_end >= 5:
        t = t[:first_end].strip()

    low = t.lower()
    refusal_triggers = [
        "i don't know", "i do not know", "unknown",
        "cannot answer", "no idea"
    ]
    if any(p in low for p in refusal_triggers):
        return "Не знаю"

    # Режим перевода
    if "переведи на английский" in (query or "").lower():
        t = t.replace('"', '').replace("'", "").replace("«", "").replace("»", "").strip()
        m = re.findall(r"[A-Za-z][A-Za-z .,!?'\-]*", t)
        if m:
            eng = m[0].strip().strip(" .")
            return eng if eng else "Не знаю"
        return "Не знаю"

    # Более мягкий лимит по словам
    words = t.split()
    if len(words) > 20:
        t = " ".join(words[:20]).rstrip(",;:")
        if not t.endswith('.'):
            t += '.'

    return t or "Не знаю"


# удалено: дубли generate_once/generate_batch (вариант на 24 токена) — используем основной и RAG-версии


# === Дополнения: строгий фолбэк и исправление "сломанных" ответов ===

STRICT_FALLBACK_PROMPT = (
    "Ты — краткий и точный ассистент.\n"
    "Правила:\n"
    "1) Одно завершённое предложение (до 14 слов).\n"
    "2) Не повторяй вопрос, не добавляй ролей.\n"
    "3) Если не уверен или вопрос абсурден — ответь: 'Не знаю'.\n"
    "4) Формулы допускаются; можно кратко расшифровать символы: 'где …'.\n"
    "5) Для переводов отвечай только переводом, без кавычек."
)

def build_strict_prompt(user_text: str) -> str:
    messages = [
        {"role": "system", "content": STRICT_FALLBACK_PROMPT},
        {"role": "user", "content": user_text}
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f"[SYSTEM]\n{STRICT_FALLBACK_PROMPT}\n\n[USER]\n{user_text}\n\n[ASSISTANT] "

def _is_broken(ans: str) -> bool:
    if not ans or not ans.strip():
        return True
    t = ans.strip()
    low = t.lower()
    words = [w for w in re.findall(r"\w+", low)]
    letters = sum(1 for c in t if c.isalpha())
    if t.endswith("?"):
        return True
    if re.search(r"(?:^|\s)не\.?$", low):
        return True
    if len(t) <= 2 or letters <= 2:
        return True
    if len(words) <= 1 and not re.fullmatch(r"\d+[\.,]?\d*", t):
        return True
    if low in {"энергии?", "тяжести?", "?", "ла?", "формула воды?"}:
        return True
    return False

def _maybe_answer_math(query: str):
    if not isinstance(query, str):
        return None
    q = query.strip().lower()
    import math as _m

    def _to_float(s: str) -> float:
        return float(s.replace(',', '.'))

    def _fmt(val: float) -> str:
        if abs(val - int(round(val))) < 1e-9:
            return f"{int(round(val))}."
        s = f"{val:.10g}"
        if "." in s:
            s = s.replace(".", ",")
        return f"{s}."

    m = re.search(r"сколько\s+будет\s+(\d+)\s*(плюс|минус|умножить\s+на|разделить\s+на|\/|:|\*|x|×|\+|\-)\s*(\d+)", q)
    if m:
        a = float(m.group(1)); op = m.group(2); b = float(m.group(3))
        if op in ("плюс", "+"): return _fmt(a + b)
        if op in ("минус", "-"): return _fmt(a - b)
        if op in ("умножить на", "*", "x", "×"): return _fmt(a * b)
        if op in ("разделить на", "/", ":"):
            if b == 0: return "Не знаю"
            return _fmt(a / b)

    m = re.search(r"(\d+)\s*(разделить\s+на|\/|:)\s*(\d+)", q)
    if m:
        a = float(m.group(1)); b = float(m.group(3))
        if b == 0: return "Не знаю"
        return _fmt(a / b)

    m = re.search(r"(\d+)\s*(умножить\s+на|\*|x|×)\s*(\d+)", q)
    if m:
        a = float(m.group(1)); b = float(m.group(3))
        return _fmt(a * b)

    m = re.search(r"сумма\s+чисел\s+(\d+)\s*и\s*(\d+)", q)
    if m:
        a = float(m.group(1)); b = float(m.group(2))
        return _fmt(a + b)

    m = re.search(r"сколько\s+будет\s+(\d+)\s*(минус)\s*(\d+)", q)
    if m:
        a = float(m.group(1)); b = float(m.group(3))
        return _fmt(a - b)

    m = re.search(r"(\d+)\s*процент(?:а|ов)?\s*от\s*(\d+)", q)
    if m:
        p = float(m.group(1)); base = float(m.group(2))
        return _fmt(base * p / 100.0)

    m = re.search(r"среднее\s+арифметическое\s+чисел\s+([\d,\s]+)", q)
    if m:
        nums = [int(s) for s in re.findall(r"\d+", m.group(1))]
        if nums:
            return _fmt(sum(nums) / len(nums))

    m = re.search(r"квадратн[а-яё]*\s+корень\s+из\s+(\d+)", q)
    if m:
        n = int(m.group(1)); r = _m.isqrt(n)
        if r * r == n:
            return _fmt(r)
        return "Не знаю"

    m = re.search(r"(\d+)\s+в\s+([а-яё]+)\s+степени", q)
    if m:
        base = int(m.group(1)); word = m.group(2)
        ord_map = {'второй':2,'третьей':3,'четвёртой':4,'четвертой':4,'пятой':5,'шестой':6,'седьмой':7,'восьмой':8,'девятой':9,'десятой':10}
        exp = ord_map.get(word)
        if exp is not None:
            return _fmt(float(base ** exp))

    m = re.search(r"площадь\s+круга\s+радиуса\s+(\d+(?:[\.,]\д+)?)", q)
    if m:
        r = _to_float(m.group(1))
        mpi = re.search(r"π\s*=\s*(\d+(?:[\.,]\д+)?)", q)
        pi = _to_float(mpi.group(1)) if mpi else 3.1415926535
        return _fmt(pi * (r ** 2))

    return None

def _generate_raw(query: str, max_new_tokens: int = 32, strict: bool = False):
    prompt = build_strict_prompt(query) if strict else build_prompt(query)
    data = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)
    data = {k: v.to(model.device) for k, v in data.items()}
    data.pop('token_type_ids', None)

    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    eos_id = tokenizer.eos_token_id or tokenizer.pad_token_id

    start = time.perf_counter()
    out_ids = model.generate(
        **data,
        generation_config=generation_config,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=getattr(generation_config, 'temperature', 0.7) or 0.7,
        top_p=getattr(generation_config, 'top_p', 0.9) or 0.9,
        pad_token_id=pad_id,
        eos_token_id=eos_id,
    )[0]
    gen_time = time.perf_counter() - start

    new_ids = out_ids[len(data['input_ids'][0]):]
    raw = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    # чистим эхо/assistant маркеры
    t = raw
    if isinstance(query, str) and t.lower().startswith(query.strip().lower()):
        t = t[len(query):].lstrip(" \\–—-:.,!?\"'\n\t")
    lines = [l.strip() for l in t.splitlines() if l.strip()]
    while lines and lines[0].lower().startswith('assistant'):
        lines.pop(0)
    t = lines[0] if lines else ""
    while t.lower().rstrip().endswith('assistant:') or t.lower().rstrip().endswith('assistant'):
        tt = t.rstrip(); ll = tt.lower()
        if ll.endswith('assistant:'):
            t = tt[:-len('assistant:')]
        elif ll.endswith('assistant'):
            t = tt[:-len('assistant')]
        else:
            break
        t = t.rstrip()

    toks = new_ids.shape[-1]
    toks_per_s = toks / max(gen_time, 1e-6)
    return t, toks, gen_time, toks_per_s

# Переопределяем generate_once с быстрым путём и фолбэком
@torch.inference_mode()
def generate_once(query: str,
                  max_new_tokens: int = 32,
                  seed: int = 127):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Быстрый путь: математика
    m = _maybe_answer_math(query)
    if m is not None:
        return m, 0, 0.0, 0.0

    text, toks, t, rate = _generate_raw(query, max_new_tokens=max_new_tokens, strict=False)
    text = _normalize_text(query, text)

    if _is_broken(text):
        text2, toks2, t2, rate2 = _generate_raw(query, max_new_tokens=max_new_tokens, strict=True)
        text2 = _normalize_text(query, text2)
        if _is_broken(text2):
            total_toks = toks + toks2
            total_time = t + t2
            total_rate = 0.0 if total_time <= 1e-6 else total_toks / total_time
            return "Не знаю", total_toks, total_time, total_rate
        else:
            total_toks = toks + toks2
            total_time = t + t2
            total_rate = 0.0 if total_time <= 1e-6 else total_toks / total_time
            return text2, total_toks, total_time, total_rate

    return text, toks, t, rate

# Переопределяем generate_batch: быстрый путь + точечный фолбэк
@torch.inference_mode()
def generate_batch(queries, batch_size: int = 8, max_new_tokens: int = 32, seed: int = 127):
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        try:
            tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})
            model.resize_token_embeddings(len(tokenizer))
        except Exception:
            pass
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    eos_id = tokenizer.eos_token_id or tokenizer.pad_token_id

    all_texts, all_tokens, all_times, all_rates = [], [], [], []

    for start in range(0, len(queries), batch_size):
        chunk = queries[start:start + batch_size]

        # Быстрые ответы по математике
        quick_answers = {}
        to_generate = []
        idx_map = []
        for i, q in enumerate(chunk):
            m = _maybe_answer_math(q)
            if m is not None:
                quick_answers[i] = m
            else:
                to_generate.append(q)
                idx_map.append(i)

        # Генерация для оставшихся
        toks_by_i = {}
        batch_time = 0.0
        if to_generate:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            prompts = [build_prompt(q) for q in to_generate]
            enc = tokenizer(
                prompts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                add_special_tokens=False,
            )
            attention_mask = enc.get('attention_mask')
            if attention_mask is not None:
                input_lengths = attention_mask.sum(dim=1)
            else:
                input_lengths = torch.tensor([enc['input_ids'].shape[1]] * enc['input_ids'].shape[0])

            enc = {k: v.to(model.device) for k, v in enc.items()}
            enc.pop('token_type_ids', None)

            start_time = time.perf_counter()
            out = model.generate(
                **enc,
                generation_config=generation_config,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=getattr(generation_config, 'temperature', 0.7) or 0.7,
                top_p=getattr(generation_config, 'top_p', 0.9) or 0.9,
                pad_token_id=pad_id,
                eos_token_id=eos_id,
            )
            batch_time = time.perf_counter() - start_time

            decoded = []
            for j in range(out.shape[0]):
                new_ids = out[j, int(input_lengths[j]):]
                raw = tokenizer.decode(new_ids, skip_special_tokens=True)
                t = raw.strip()
                q = to_generate[j]
                if isinstance(q, str) and t.lower().startswith(q.strip().lower()):
                    t = t[len(q):].lstrip(" \\–—-:.,!?\"'\n\t")
                lines = [l.strip() for l in t.splitlines() if l.strip()]
                while lines and lines[0].lower().startswith('assistant'):
                    lines.pop(0)
                t = lines[0] if lines else ""
                while t.lower().rstrip().endswith('assistant:') or t.lower().rstrip().endswith('assistant'):
                    tt = t.rstrip(); ll = tt.lower()
                    if ll.endswith('assistant:'):
                        t = tt[:-len('assistant:')]
                    elif ll.endswith('assistant'):
                        t = tt[:-len('assistant')]
                    else:
                        break
                    t = t.rstrip()
                toks_by_i[idx_map[j]] = int(new_ids.shape[-1])
                decoded.append(t)

        # Сбор ответов с нормализацией и фолбэком
        for i, q in enumerate(chunk):
            if i in quick_answers:
                text = quick_answers[i]
                toks = 0
                t_time = 0.0
                rate = 0.0
            else:
                t0 = decoded[idx_map.index(i)] if i in idx_map else ""
                text = _normalize_text(q, t0)
                toks = toks_by_i.get(i, 0)
                t_time = batch_time
                rate = 0.0 if batch_time <= 1e-6 else toks / batch_time

                if _is_broken(text):
                    t2, toks2, tt2, rr2 = _generate_raw(q, max_new_tokens=max_new_tokens, strict=True)
                    t2 = _normalize_text(q, t2)
                    if _is_broken(t2):
                        text = "Не знаю"
                    else:
                        text = t2
                    toks += toks2
                    t_time += tt2
                    rate = 0.0 if t_time <= 1e-6 else toks / t_time

            all_texts.append(text)
            all_tokens.append(toks)
            all_times.append(t_time)
            all_rates.append(rate)

    return all_texts, all_tokens, all_times, all_rates

In [ ]:
# Установка FAISS для Colab (с фолбэком на CPU)
import os
import torch

if torch.cuda.is_available():
    # На L4/T4 Colab эта команда должна найти совместимую версию
    print("Установка faiss-gpu...")
    os.system("pip install faiss-gpu")
else:
    print("CUDA не найдена. Установка faiss-cpu...")
    os.system("pip install faiss-cpu")

# Проверяем импорт
try:
    import faiss
    print("✅ FAISS успешно импортирован.")
except ImportError:
    print("🔴 FAISS не удалось импортировать. Пожалуйста, перезапустите среду выполнения (Runtime -> Restart runtime).")

Установка faiss-gpu...
✅ FAISS успешно импортирован.


In [ ]:
# === RAG: загрузка индекса FAISS и ретривер ===
import json, re
from pathlib import Path
from bisect import bisect_right
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer

try:
    import faiss
except ImportError as e:
    raise ImportError("FAISS не найден. Установите faiss-cpu (Windows) или faiss-gpu (Linux) и перезапустите ядро.") from e

# Пути к артефактам
RAG_DIR = Path('rag')
META_PATH = RAG_DIR / 'meta.json'
INDEX_PATH = RAG_DIR / 'index.faiss'
CHUNKS_PATH = RAG_DIR / 'chunks.parquet'

# Загрузка метаданных и компонентов
with open(META_PATH, 'r', encoding='utf-8') as f:
    RAG_META = json.load(f)
EMB_MODEL_NAME = RAG_META.get('model', 'sentence-transformers/all-MiniLM-L12-v2')
EMB_DIM = int(RAG_META.get('dim', 384))

retr_model = SentenceTransformer(EMB_MODEL_NAME, device='cuda' if torch.cuda.is_available() else 'cpu')
faiss_index = faiss.read_index(str(INDEX_PATH))
pf = pq.ParquetFile(CHUNKS_PATH)

# Препроцесс
_re_punct = re.compile(r"[\,\.;:!\?\(\)\[\]\{\}\-\—\–\«\»\"\']+")
_re_space = re.compile(r"\s+")
try:
    from lem_worker import lemmatize_text as _lemmatize_ru
except Exception:
    def _lemmatize_ru(s: str) -> str:
        return s  # фолбэк без лемматизации

def _norm(s: str) -> str:
    s = s.lower().strip()
    s = _re_punct.sub(' ', s)
    s = _re_space.sub(' ', s)
    return s.strip()

def _norm_lemma(s: str) -> str:
    return _lemmatize_ru(_norm(s))

# Индексация row-groups для выборки по индексам
_rg_sizes = [pf.metadata.row_group(i).num_rows for i in range(pf.metadata.num_row_groups)]
_cum = []
acc = 0
for n in _rg_sizes:
    acc += n
    _cum.append(acc)

def _rows_by_indices(indices):
    # Группируем индексы по row-group
    grp_map = {}
    for idx in indices:
        if idx < 0:
            continue
        g = bisect_right(_cum, idx)
        prev = 0 if g == 0 else _cum[g-1]
        rel = idx - prev
        d = grp_map.setdefault(g, {'prev': prev, 'rels': []})
        d['rels'].append(rel)
    # Читаем только нужные группы
    out = {}
    for g, info in grp_map.items():
        prev = info['prev']
        rels = info['rels']
        tbl = pf.read_row_group(g, columns=['title', 'text_lem'])
        for rel in rels:
            if 0 <= rel < tbl.num_rows:
                abs_idx = prev + rel
                out[abs_idx] = (
                    tbl.column('title')[rel].as_py(),
                    tbl.column('text_lem')[rel].as_py(),
                )
    return [out.get(i, ('', '')) for i in indices]

DECISION_THRESHOLD = 0.38
TOP_K = 3
MAX_CTX_CHARS = 1400

def retrieve_context(query: str, k: int = TOP_K):
    qn = _norm_lemma(query)
    vec = retr_model.encode([qn], normalize_embeddings=True).astype('float32')
    D, I = faiss_index.search(vec, k)
    indices = I[0].tolist()
    scores = [float(x) for x in D[0]]
    pairs = _rows_by_indices(indices)
    per = max(200, MAX_CTX_CHARS // max(1, k))
    parts = []
    for (title, text), sc in zip(pairs, scores):
        snippet = (text or '')[:per]
        parts.append(snippet)
    ctx = "\n\n".join(parts)
    return scores, indices, ctx

NEW_SYSTEM_PROMPT = (
    "Ты — краткий и точный ассистент на русском языке.\n\n"
    "СТРОГИЕ ПРАВИЛА:\n"
    "1. Отвечай одним коротким предложением (до 10 слов).\n"
    "2. Используй ТОЛЬКО предоставленный контекст.\n"
    "3. Если в контексте нет ответа — отвечай ровно: 'Не знаю'.\n"
    "4. Не рассуждай, не объясняй, не добавляй лишних деталей.\n"
    "5. Не повторяй вопрос и не добавляй префиксы ролей.\n"
)

def _build_rag_prompt(question: str, ctx: str) -> str:
    user = (
        "Используй приведённый контекст для ответа. Если в контексте нет ответа — ответь: 'Не знаю'.\n"
        f"Контекст:\n{ctx}\n\nВопрос: {question}"
    )
    try:
        messages = [
            {"role": "system", "content": NEW_SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f"[SYSTEM]\n{NEW_SYSTEM_PROMPT}\n\n[USER]\n{user}\n\n[ASSISTANT] "



In [ ]:
# === RAG-генерация с Decision-порогом ===
@torch.inference_mode()
def generate_once_rag(query: str, max_new_tokens: int = 32, seed: int = 127):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Быстрый путь: математика
    m = _maybe_answer_math(query)
    if m is not None:
        return m, 0, 0.0, 0.0

    scores, idxs, ctx = retrieve_context(query, k=TOP_K)
    best = max(scores) if scores else 0.0
    if not scores or best < DECISION_THRESHOLD or not ctx.strip():
        return "Не знаю", 0, 0.0, 0.0

    prompt = _build_rag_prompt(query, ctx)
    data = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)
    data = {k: v.to(model.device) for k, v in data.items()}
    data.pop('token_type_ids', None)

    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    eos_id = tokenizer.eos_token_id or tokenizer.pad_token_id

    start = time.perf_counter()
    out_ids = model.generate(
        **data,
        generation_config=generation_config,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=getattr(generation_config, 'temperature', 0.7) or 0.7,
        top_p=getattr(generation_config, 'top_p', 0.9) or 0.9,
        pad_token_id=pad_id,
        eos_token_id=eos_id,
    )[0]
    gen_time = time.perf_counter() - start

    new_ids = out_ids[len(data['input_ids'][0]):]
    raw = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    text = _normalize_text(query, raw)

    if _is_broken(text):
        return "Не знаю", int(new_ids.shape[-1]), gen_time, 0.0

    toks = int(new_ids.shape[-1])
    rate = toks / max(gen_time, 1e-6)
    return text, toks, gen_time, rate


@torch.inference_mode()
def generate_batch_rag(queries, batch_size: int = 4, max_new_tokens: int = 32, seed: int = 127):
    tokenizer.padding_side = 'left'
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    eos_id = tokenizer.eos_token_id or tokenizer.pad_token_id

    all_texts, all_tokens, all_times, all_rates = [], [], [], []

    for start in range(0, len(queries), batch_size):
        chunk = queries[start:start + batch_size]

        # Предварительно получаем контексты и решаем про отказ
        prompts = []
        idx_map = []
        for i, q in enumerate(chunk):
            m = _maybe_answer_math(q)
            if m is not None:
                all_texts.append(m); all_tokens.append(0); all_times.append(0.0); all_rates.append(0.0)
                continue
            scores, ids, ctx = retrieve_context(q, k=TOP_K)
            best = max(scores) if scores else 0.0
            if not scores or best < DECISION_THRESHOLD or not ctx.strip():
                all_texts.append("Не знаю"); all_tokens.append(0); all_times.append(0.0); all_rates.append(0.0)
            else:
                prompts.append(_build_rag_prompt(q, ctx))
                idx_map.append(i)

        if not prompts:
            continue

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        enc = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            add_special_tokens=False,
        )
        input_lengths = enc['attention_mask'].sum(dim=1) if 'attention_mask' in enc else torch.tensor([enc['input_ids'].shape[1]] * enc['input_ids'].shape[0])
        enc = {k: v.to(model.device) for k, v in enc.items()}
        enc.pop('token_type_ids', None)

        start_time = time.perf_counter()
        out = model.generate(
            **enc,
            generation_config=generation_config,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=getattr(generation_config, 'temperature', 0.7) or 0.7,
            top_p=getattr(generation_config, 'top_p', 0.9) or 0.9,
            pad_token_id=pad_id,
            eos_token_id=eos_id,
        )
        batch_time = time.perf_counter() - start_time

        for j in range(out.shape[0]):
            new_ids = out[j, int(input_lengths[j]):]
            raw = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            text = _normalize_text("", raw)
            if _is_broken(text):
                text = "Не знаю"
            toks = int(new_ids.shape[-1])
            rate = 0.0 if batch_time <= 1e-6 else toks / batch_time
            # вставляем в позицию исходного запроса
            pos = idx_map[j]
            all_texts.insert(start + pos - start, text)
            all_tokens.insert(start + pos - start, toks)
            all_times.insert(start + pos - start, batch_time)
            all_rates.insert(start + pos - start, rate)

    return all_texts, all_tokens, all_times, all_rates



In [ ]:
TEST_QUERIES = [

# === 1. ПЕРЕФОРМУЛИРОВКИ ОДНОГО ВОПРОСА ===
"Кто является создателем языка программирования Python?",
"Как зовут человека, который разработал Python?",
"Кто придумал язык программирования Python?",
"Кто автор Python?",
"Чьим детищем является язык программирования Python?",

# === 2. ПЕРЕФОРМУЛИРОВКИ СЛОЖНОГО ВОПРОСА ===
"В каком году была опубликована первая версия языка программирования Rust?",
"Когда впервые был представлен язык Rust?",
"Когда вышла первая стабильная версия Rust?",
"В каком году мир увидел язык программирования Rust?",

# === 3. СЛОЖНЫЕ ФАКТИЧЕСКИЕ ВОПРОСЫ ===
"Каков был первоначальный девиз компании Google до смены на 'Don't be evil'?",
"Какой химический элемент был открыт первым в результате искусственного синтеза?",
"Какой была первая книга, напечатанная Иоганном Гутенбергом на своем печатном станке?",
"Кто из римских императоров написал философский труд 'Размышления' на греческом языке?",
"Какой элемент периодической таблицы имеет самую высокую температуру плавления?",

# === 4. СПЕЦИФИЧЕСКИЕ ЗНАНИЯ ===
"Какой протокол используется для автоматической настройки сетевых параметров IP-адресации?",
"Как называется самая глубокая известная пещера в мире и где она расположена?",
"Какой химический процесс лежит в основе работы литий-ионных аккумуляторов?",
"Кто является автором концепции 'мемов' в эволюционной биологии?",

# === 5. ПРОВОКАЦИОННЫЕ ВОПРОСЫ (без корректного ответа) ===
"В каком году был изобретен первый квантовый компьютер для домашнего использования?",
"Какой химический элемент открыла Мария Кюри в 1925 году?",
"Кто из древнегреческих философов написал трактат о квантовой механике?",
"Какой российский император подписал декларацию о независимости США?",
"В каком году Чингисхан посетил Париж?",

# === 6. ВОПРОСЫ С МНОЖЕСТВЕННЫМИ СУЩНОСТЯМИ ===
"Какие два ученых независимо друг от друга открыли закон сохранения энергии?",
"Назовите трех лауреатов Нобелевской премии по физике 2020 года",
"Какие четыре спутника Юпитера открыл Галилей?",
"Какие два элемента входят в состав поваренной соли?",

# === 7. ВОПРОСЫ НА СОПОСТАВЛЕНИЕ ===
"В чем разница между RAM и ROM в компьютерной памяти?",
"Чем отличается ДНК от РНК по химическому составу?",
"Каковы основные различия между фотосинтезом и хемосинтезом?",
"Чем TCP отличается от UDP в компьютерных сетях?",

# === 8. ВОПРОСЫ С ЧИСЛОВЫМИ ОТВЕТАМИ ===
"Сколько хромосом у обыкновенной плодовой мушки Drosophila melanogaster?",
"Какова приблизительная скорость света в вакууме в метрах в секунду?",
"Сколько костей в скелете взрослого человека?",
"Каков атомный номер углерода в периодической таблице?",

# === 9. ИСТОРИЧЕСКИЕ ДАТЫ И СОБЫТИЯ ===
"В каком году произошла битва при Гастингсе?",
"Когда была подписана Великая хартия вольностей?",
"В каком году началась Первая мировая война?",
"Когда человек впервые ступил на Луну?",

# === 10. ГЕОГРАФИЧЕСКИЕ ВОПРОСЫ ===
"Какая река является самой длинной в мире?",
"Какой вулкан является самым высоким в Африке?",
"Какое озеро является самым глубоким в мире?",
"Какой пролив разделяет Европу и Африку?",

# === 11. ЛИТЕРАТУРНЫЕ ВОПРОСЫ ===
"Кто автор романа 'Сто лет одиночества'?",
"Какая пьеса Шекспира начинается с фразы 'Когда наступает час двенадцатый'?",
"Кто написал 'Божественную комедию'?",
"Какой русский писатель автор 'Мертвых душ'?",

# === 12. НАУЧНЫЕ ТЕРМИНЫ И ПОНЯТИЯ ===
"Что такое 'сингулярность' в математике?",
"Как называется процесс деления клетки?",
"Что изучает наука герпетология?",
"Какой газ составляет большую часть атмосферы Земли?",
]

In [ ]:
TEST_QUERIES_HARD = [

# === 1. МНОГОСЛОЙНЫЕ ИСТОРИЧЕСКИЕ ВОПРОСЫ ===
"Какие были основные последствия Вестфальского мира 1648 года для развития международных отношений в Европе?",
"В чем заключалась разница между подходами Аристотеля и Платона к теории идей?",
"Как повлияло изобретение книгопечатания на распространение Реформации в Европе?",

# === 2. НАУЧНЫЕ КОНЦЕПЦИИ С НЮАНСАМИ ===
"Объясните парадокс Эйнштейна-Подольского-Розена и его значение для квантовой механики",
"В чем разница между алгебраической и трансцендентной топологией?",
"Каков механизм действия ингибиторов протонной помпы и почему они эффективны при язвенной болезни?",

# === 3. ФИЛОСОФСКИЕ И АБСТРАКТНЫЕ ПОНЯТИЯ ===
"В чем заключается различие между кантовскими категориями 'априори' и 'апостериори'?",
"Объясните концепцию 'вечного возвращения' у Ницше",
"Что такое 'фаллоцентризм' в деконструктивистской теории Деррида?",

# === 4. СПЕЦИАЛИЗИРОВАННЫЕ ТЕРМИНЫ ===
"Что такое 'хемотаксис' у бактерий и как он регулируется?",
"Объясните принцип 'трансформации признаков' в филогенетическом анализе",
"Что такое 'квантиль-квантиль график' в статистике и для чего он используется?",

# === 5. МЕЖДИСЦИПЛИНАРНЫЕ ВОПРОСЫ ===
"Как теория игр применяется в эволюционной биологии?",
"Какое влияние оказала квантовая механика на философию детерминизма?",
"Как принципы архитектуры барокко проявляются в музыке Баха?",

# === 6. КОНТРФАКТИЧЕСКИЕ ИСТОРИЧЕСКИЕ ВОПРОСЫ ===
"Какие могли быть последствия, если бы Александр Македонский не умер в 32 года?",
"Как изменилась бы научная революция, если бы Галилей отрекся от своих взглядов?",
"Что если бы Наполеон выиграл битву при Ватерлоо?",

# === 7. СЛОЖНЫЕ ЛИТЕРАТУРНЫЕ АНАЛИЗЫ ===
"Как тема 'двойничества' раскрывается в 'Братьях Карамазовых' Достоевского?",
"В чем заключается новаторство потока сознания в 'Улиссе' Джойса?",
"Как мифологический подтекст влияет на структуру 'Волхва' Фаулза?",

# === 8. ЮРИДИЧЕСКИЕ КАЗУСЫ ===
"В чем разница между 'actus reus' и 'mens rea' в уголовном праве?",
"Как применяется принцип 'добросовестности' в международном коммерческом арбитраже?",
"Что такое 'доктрина обнаружения' в праве интеллектуальной собственности?",

# === 9. ЭКОНОМИЧЕСКИЕ ТЕОРИИ ===
"В чем заключаются основные различия между монетаризмом Фридмана и кейнсианством?",
"Как теория сравнительных преимуществ Рикардо применяется в современной международной торговле?",
"Что такое 'парадокс Триффина' и как он проявляется сегодня?",

# === 10. ЛИНГВИСТИЧЕСКИЕ СЛОЖНОСТИ ===
"Объясните разницу между эргативным и номинативно-аккузативным строем языка",
"Что такое 'глоттогония' и какие существуют теории о праиндоевропейском языке?",
"Как работает грамматика в языках с полисинтетическим строем?",

# === 11. ИСКУССТВОВЕДЧЕСКИЕ НЮАНСЫ ===
"В чем заключался художественный метод 'дивизионизма' у Сёра?",
"Как принципы супрематизма Малевича повлияли на архитектуру авангарда?",
"Что такое 'редимейд' в концептуализме Дюшана и как это изменило понимание искусства?",

# === 12. СОВРЕМЕННЫЕ ТЕХНОЛОГИЧЕСКИЕ ВОПРОСЫ ===
"Объясните принцип работы гомоморфного шифрования и его практическое применение",
"В чем разница между трансформером и LSTM в обработке естественного языка?",
"Как работает proof-of-stake в сравнении с proof-of-work в блокчейне?",

# === 13. МЕДИЦИНСКИЕ ДИЛЕММЫ ===
"В чем заключается этическая проблема 'двойного эффекта' в паллиативной медицине?",
"Как работает механизм резистентности к ингибиторам тирозинкиназы при лечении рака?",
"Что такое 'синдром отмены' у антидепрессантов SSRI и как его минимизировать?",

# === 14. МАТЕМАТИЧЕСКИЕ КОНЦЕПЦИИ ===
"Объясните теорему Гёделя о неполноте и ее значение для оснований математики",
"В чем заключается гипотеза Римана и почему она важна для теории чисел?",
"Что такое 'топологическая размерность' и как она отличается от хаусдорфовой размерности?",

# === 15. ПСИХОЛОГИЧЕСКИЕ ТЕОРИИ ===
"В чем разница между классическим и оперантным обусловливанием в бихевиоризме?",
"Как теория привязанности Боулби объясняет формирование личностных расстройств?",
"Что такое 'метакогнитивная терапия' и как она применяется при депрессии?",

# === 16. КУЛЬТУРОЛОГИЧЕСКИЕ КОНЦЕПЦИИ ===
"Объясните концепцию 'воображаемых сообществ' Андерсона и ее применение",
"В чем заключается разница между 'высокой' и 'низкой' контекстностью культур по Холлу?",
"Что такое 'культурный гегемонизм' у Грамши и как он проявляется сегодня?",

# === 17. ЭКОЛОГИЧЕСКИЕ СИСТЕМЫ ===
"Как работает концепция 'трофического каскада' в экологии?",
"В чем разница между альфа-, бета- и гамма-разнообразием в биологии?",
"Что такое 'экологическая сукцессия' и какие существуют ее типы?",

# === 18. МУЗЫКАЛЬНАЯ ТЕОРИЯ ===
"Объясните принципы додекафонии Шёнберга и их влияние на современную музыку",
"В чем разница между модальной и тональной гармонией?",
"Что такое 'спектральная музыка' и какие у нее характерные черты?",

# === 19. АРХИТЕКТУРНЫЕ КОНЦЕПЦИИ ===
"Как принципы 'органической архитектуры' Райта реализованы в 'Доме над водопадом'?",
"В чем заключалась новаторская концепция 'променад архитекurale' у Ле Корбюзье?",
"Что такое 'биофильный дизайн' в современной архитектуре?",

# === 20. СЛОЖНЫЕ ЭТИЧЕСКИЕ ДИЛЕММЫ ===
"В чем заключается 'проблема вагонетки' в этике и какие у нее варианты?",
"Как теория справедливости Ролза применяется к современному неравенству?",
"Что такое 'этика добродетели' Аристотеля и чем она отличается от деонтологии?",
]

NUM_QUERIES_HARD = len(TEST_QUERIES_HARD)
print({'num_queries': NUM_QUERIES_HARD})

{'num_queries': 60}


In [ ]:
# === Тест RAG/без RAG на подмножестве ===
USE_RAG = True  # переключатель RAG (по умолчанию выкл), можно включить при необходимости
USE_BATCH = True
BATCH_SIZE = 6
SAMPLE_N = 60

subset = TEST_QUERIES
results = []

if USE_BATCH:
    if USE_RAG:
        texts, tokens_list, times_list, rates_list = generate_batch_rag(
            subset, batch_size=BATCH_SIZE, max_new_tokens=64, seed=127
        )
    else:
        texts, tokens_list, times_list, rates_list = generate_batch(
            subset, batch_size=BATCH_SIZE, max_new_tokens=64, seed=127
        )
    for q, text, toks, t, rate in zip(subset, texts, tokens_list, times_list, rates_list):
        print(f"\nQ: {q}\nA: {text}\n")
        results.append({
            'query': q,
            'tokens': toks,
            'time_s': round(t, 3),
            'tok_s': round(rate, 2),
            'answer': text[:500]
        })
else:
    for q in tqdm(subset, desc='Inference', unit='q'):
        if USE_RAG:
            text, toks, t, rate = generate_once_rag(q, max_new_tokens=64, seed=127)
        else:
            text, toks, t, rate = generate_once(q, max_new_tokens=64, seed=127)
        print(f"\nQ: {q}\nA: {text}\n")
        results.append({
            'query': q,
            'tokens': toks,
            'time_s': round(t, 3),
            'tok_s': round(rate, 2),
            'answer': text[:500]
        })

print({'mode': load_mode,
       'rag': USE_RAG,
       'avg_tok_s': round(mean([r['tok_s'] for r in results]) if results else 0.0, 2),
       'avg_time_s': round(mean([r['time_s'] for r in results]) if results else 0.0, 2),
       'count': len(results)})



Q: Кто является создателем языка программирования Python?
A: Создатель языка программирования Python — Гвидо ван Россум.


Q: Как зовут человека, который разработал Python?
A: Гвидо ван Россум (Guido van Rossum).


Q: Кто придумал язык программирования Python?
A: Язык программирования Python был придуман Гвидо ван Россумом (Guido van Rossum).


Q: Кто автор Python?
A: Автором языка программирования Python является Гвидо ван Россум.


Q: Чьим детищем является язык программирования Python?
A: Язык программирования Python был создан Гвидо ван Россумом.


Q: В каком году была опубликована первая версия языка программирования Rust?
A: Первая стабильная версия языка программирования Rust, 0.


Q: Когда впервые был представлен язык Rust?
A: Язык программирования Rust был представлен в 2010 году, но его разработка началась раньше, примерно с конца 2006 года.


Q: Когда вышла первая стабильная версия Rust?
A: Первая стабильная версия языка программирования Rust, 0.


Q: В каком году мир увидел

In [ ]:
# Вывод таблицы результатов и примеров ответов (по results)
from tabulate import tabulate
headers = ["#", "tok/s", "time_s", "tokens", "query", "answer"]
rows = [[i, r['tok_s'], r['time_s'], r['tokens'], r['query'], r['answer']]
        for i, r in enumerate(results, 1)]
print(tabulate(rows, headers=headers, tablefmt="github"))


|   # |   tok/s |   time_s |   tokens | query                                                                                     | answer                                                                                                                                                                |
|-----|---------|----------|----------|-------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------|
|   1 |    5.5  |    6.361 |       35 | Кто является создателем языка программирования Python?                                    | Создатель языка программирования Python — Гвидо ван Россум.                                                                                                           |
|   2 |    5.66 |    6.361 |       36 | Как зовут человека, который разработал Python?              